In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(run_name="test_installation"):
    mlflow.log_param("test", 1)
    mlflow.log_metric("pr_auc", 0.5)

print("MLflow OK, version", mlflow.__version__)

In [0]:
from pyspark.sql import functions as F

SOURCE = "/Volumes/olist_dev/raw/source_olist"

FICHIERS = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

def lire_csv(nom_fichier):
    return (
        spark.read
        .option("header", True)
        .option("multiLine", True)   # avis sur plusieurs lignes
        .option("escape", '"')       # guillemets doublés dans le texte
        .option("inferSchema", True) # OK pour explorer, interdit en Bronze
        .csv(f"{SOURCE}/{nom_fichier}")
    )

dfs = {nom: lire_csv(fichier) for nom, fichier in FICHIERS.items()}

for nom, df in dfs.items():
    print(f"{nom:<22} {df.count():>10,} lignes  {len(df.columns):>3} colonnes")

In [0]:
CLES = {
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"],
    "reviews": ["review_id"],
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "products": ["product_id"],
    "category_translation": ["product_category_name"],
}

for nom, cle in CLES.items():
    df = dfs[nom]
    total = df.count()
    distinctes = df.select(*cle).distinct().count()
    nulles = df.filter(" OR ".join(f"{c} IS NULL" for c in cle)).count()
    statut = "OK" if total == distinctes and nulles == 0 else "⚠️ À TRAITER"
    print(f"{nom:<22} total={total:>8,}  distinctes={distinctes:>8,}  nulles={nulles:>4}  {statut}")

In [0]:
for nom, df in dfs.items():
    print(f"\n{'=' * 20} {nom} ({df.count():,} lignes) {'=' * 20}")
    df.printSchema()
    display(df.limit(5))

In [0]:
orders = dfs["orders"]

display(orders.groupBy("order_status").count().orderBy(F.desc("count")))

livrees = orders.filter(
    (F.col("order_status") == "delivered")
    & F.col("order_delivered_customer_date").isNotNull()
)

cible = livrees.withColumn(
    "en_retard",
    (F.to_date("order_delivered_customer_date") > F.to_date("order_estimated_delivery_date")).cast("int"),
)

display(cible.agg(
    F.count("*").alias("commandes_livrees"),
    F.sum("en_retard").alias("retards"),
    F.round(F.avg("en_retard") * 100, 2).alias("taux_retard_pct"),
))

In [0]:
par_mois = (
    cible.groupBy(F.date_trunc("month", "order_purchase_timestamp").alias("mois"))
    .agg(
        F.count("*").alias("commandes"),
        F.round(F.avg("en_retard") * 100, 1).alias("taux_retard_pct"),
    )
    .orderBy("mois")
)
display(par_mois)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
fin = (
    orders.filter(F.col("order_purchase_timestamp") >= "2018-06-01")
    .groupBy(F.date_trunc("week", "order_purchase_timestamp").alias("semaine"))
    .agg(
        F.count("*").alias("commandes"),
        F.round(F.avg((F.col("order_status") == "delivered").cast("int")) * 100, 1).alias("pct_livrees"),
    )
    .orderBy("semaine")
)
display(fin)

In [0]:
controles = {
    "approbation manquante": F.col("order_approved_at").isNull(),
    "approuvée avant l'achat": F.col("order_approved_at") < F.col("order_purchase_timestamp"),
    "transporteur avant approbation": F.col("order_delivered_carrier_date") < F.col("order_approved_at"),
    "livrée avant transporteur": F.col("order_delivered_customer_date") < F.col("order_delivered_carrier_date"),
    "delivered sans date de livraison": (F.col("order_status") == "delivered") & F.col("order_delivered_customer_date").isNull(),
    "date de livraison mais pas delivered": (F.col("order_status") != "delivered") & F.col("order_delivered_customer_date").isNotNull(),
}

total = orders.count()
for nom, condition in controles.items():
    n = orders.filter(condition).count()
    print(f"{nom:<40} {n:>7,}  ({n / total * 100:.2f} %)")

In [0]:
signaux = (
    cible
    .join(dfs["customers"].select("customer_id", "customer_state"), "customer_id")
    .withColumn("delai_promis_j", F.datediff("order_estimated_delivery_date", "order_purchase_timestamp"))
)

# Le retard selon l'État du client
display(
    signaux.groupBy("customer_state")
    .agg(F.count("*").alias("commandes"), F.round(F.avg("en_retard") * 100, 1).alias("taux_retard_pct"))
    .orderBy(F.desc("taux_retard_pct"))
)

# Le retard selon le délai promis au client
display(
    signaux.groupBy((F.floor(F.col("delai_promis_j") / 7) * 7).alias("delai_promis_j"))
    .agg(F.count("*").alias("commandes"), F.round(F.avg("en_retard") * 100, 1).alias("taux_retard_pct"))
    .orderBy("delai_promis_j")
)

In [0]:
items, sellers, products = dfs["order_items"], dfs["sellers"], dfs["products"]

# Une coordonnée par code postal : médiane des points situés au Brésil
geo_zip = (
    dfs["geolocation"]
    .filter(F.col("geolocation_lat").between(-33.75, 5.27) & F.col("geolocation_lng").between(-73.99, -34.79))
    .groupBy(F.col("geolocation_zip_code_prefix").alias("zip"))
    .agg(
        F.expr("percentile_approx(geolocation_lat, 0.5)").alias("lat"),
        F.expr("percentile_approx(geolocation_lng, 0.5)").alias("lng"),
    )
)

def distance_km(lat1, lng1, lat2, lng2):
    """Distance à vol d'oiseau (formule de haversine), en km."""
    # Accepte des noms de colonnes ("lat_c") ou des colonnes (F.col("lat_c"))
    lat1, lng1, lat2, lng2 = [F.col(c) if isinstance(c, str) else c for c in (lat1, lng1, lat2, lng2)]
    dlat, dlng = F.radians(lat2 - lat1), F.radians(lng2 - lng1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlng / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))
    
clients = (
    dfs["customers"].select("customer_id", "customer_state", F.col("customer_zip_code_prefix").alias("zip"))
    .join(geo_zip, "zip", "left")
    .select("customer_id", "customer_state", F.col("lat").alias("lat_c"), F.col("lng").alias("lng_c"))
)
vendeurs = (
    sellers.select("seller_id", "seller_state", F.col("seller_zip_code_prefix").alias("zip"))
    .join(geo_zip, "zip", "left")
    .select("seller_id", "seller_state", F.col("lat").alias("lat_v"), F.col("lng").alias("lng_v"))
)

# Grain : un article d'une commande
articles = (
    items
    .join(orders.select("order_id", "customer_id"), "order_id")
    .join(clients, "customer_id")
    .join(vendeurs, "seller_id")
    .join(products.select("product_id", "product_weight_g"), "product_id", "left")
    .withColumn("distance_km", distance_km("lat_v", "lng_v", "lat_c", "lng_c"))
)

# Grain : une commande
par_commande = articles.groupBy("order_id").agg(
    F.countDistinct("seller_id").alias("nb_vendeurs"),
    F.max("distance_km").alias("distance_max_km"),
    F.sum("product_weight_g").alias("poids_total_g"),
    F.min((F.col("seller_state") == F.col("customer_state")).cast("int")).alias("meme_etat"),
    F.max("shipping_limit_date").alias("date_limite_expedition"),
)

vue = (
    cible.join(par_commande, "order_id")
    .withColumn("vendeur_en_retard",
                (F.col("order_delivered_carrier_date") > F.col("date_limite_expedition")).cast("int"))
)
print(f"commandes analysées : {vue.count():,}")

In [0]:
def taux_par(colonne):
    display(
        vue.groupBy(colonne)
        .agg(F.count("*").alias("commandes"), F.round(F.avg("en_retard") * 100, 1).alias("taux_retard_pct"))
        .orderBy(colonne)
    )

# H1 : le vendeur est-il responsable ?
taux_par("vendeur_en_retard")

# H2 : vendeur et client dans le même État ?
taux_par("meme_etat")

# H3 : la distance
vue = vue.withColumn("tranche_distance",
    F.when(F.col("distance_max_km").isNull(), "z. inconnue")
     .when(F.col("distance_max_km") < 100, "a. < 100 km")
     .when(F.col("distance_max_km") < 500, "b. 100-500 km")
     .when(F.col("distance_max_km") < 1000, "c. 500-1000 km")
     .when(F.col("distance_max_km") < 2000, "d. 1000-2000 km")
     .otherwise("e. > 2000 km"))
taux_par("tranche_distance")

# H4 : plusieurs vendeurs dans une même commande
taux_par("nb_vendeurs")

# H5 : les vendeurs sont-ils très différents entre eux ?
par_vendeur = (
    items.select("order_id", "seller_id").distinct()
    .join(cible.select("order_id", "en_retard"), "order_id")
    .groupBy("seller_id").agg(F.count("*").alias("commandes"), F.avg("en_retard").alias("taux"))
    .filter("commandes >= 30")
)
display(par_vendeur.agg(
    F.count("*").alias("vendeurs_30_commandes_ou_plus"),
    F.expr("percentile_approx(taux, array(0.1, 0.5, 0.9))").alias("taux_p10_p50_p90"),
))

In [0]:
# ---------- B. Le test est-il tronqué ? ----------
display(
    orders.filter(F.col("order_purchase_timestamp") >= "2018-05-01")
    .groupBy(F.date_trunc("month", "order_purchase_timestamp").alias("mois"))
    .pivot("order_status").count()
    .orderBy("mois")
)

# ---------- C. Le taux de retard de chaque période ----------
decoupe = (
    cible
    .filter((F.col("order_purchase_timestamp") >= "2017-01-01")
            & (F.col("order_purchase_timestamp") < "2018-08-27"))
    .withColumn("periode",
        F.when(F.col("order_purchase_timestamp") < "2018-04-01", "1_entrainement")
         .when(F.col("order_purchase_timestamp") < "2018-06-01", "2_validation")
         .otherwise("3_test"))
)
display(
    decoupe.groupBy("periode")
    .agg(F.count("*").alias("commandes"), F.sum("en_retard").alias("retards"),
         F.round(F.avg("en_retard") * 100, 2).alias("taux_retard_pct"))
    .orderBy("periode")
)

# ---------- D. Le retard dégrade-t-il la satisfaction ? ----------
reviews = dfs["reviews"]
print("couples (review_id, order_id) distincts :",
      reviews.select("review_id", "order_id").distinct().count())
print("commandes avec plusieurs avis :",
      reviews.groupBy("order_id").count().filter("count > 1").count())
display(
    cible.join(reviews.select("order_id", "review_score"), "order_id")
    .groupBy("en_retard")
    .agg(F.count("*").alias("avis"),
         F.round(F.avg("review_score"), 2).alias("note_moyenne"),
         F.round(F.avg((F.col("review_score") <= 2).cast("int")) * 100, 1).alias("pct_avis_1_ou_2"))
)